In [66]:
import pandas as pd
import numpy

In [116]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB

In [67]:
df = pd.read_csv("insurance.csv")
df.head()

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category
0,67,119.8,1.56,2.92,False,Jaipur,retired,High
1,36,101.1,1.83,34.28,False,Chennai,freelancer,Low
2,39,56.8,1.64,36.64,False,Indore,freelancer,Low
3,22,109.4,1.55,3.34,True,Mumbai,student,Medium
4,69,62.2,1.60,3.94,True,Indore,retired,High


In [68]:
def ReturnUnique(col: str):
    return df[col].unique()

In [69]:
print(ReturnUnique('smoker'))
print("-" * 50)
print(ReturnUnique('city'))
print("-" * 50)
print(ReturnUnique('occupation'))

[False  True]
--------------------------------------------------
<StringArray>
[    'Jaipur',    'Chennai',     'Indore',     'Mumbai',       'Kota',
  'Hyderabad',      'Delhi', 'Chandigarh',       'Pune',    'Kolkata',
    'Lucknow',       'Gaya',  'Jalandhar',     'Mysore',  'Bangalore']
Length: 15, dtype: str
--------------------------------------------------
<StringArray>
[       'retired',     'freelancer',        'student', 'government_job',
 'business_owner',     'unemployed',    'private_job']
Length: 7, dtype: str


In [70]:
def ReturnCount(col: str):
    return df[col].value_counts()

In [71]:
print(ReturnCount('smoker'))
print("-" * 50)
print(ReturnCount('city'))
print("-" * 50)
print(ReturnCount('occupation'))

smoker
False    57
True     43
Name: count, dtype: int64
--------------------------------------------------
city
Jaipur        9
Indore        9
Chandigarh    9
Lucknow       9
Jalandhar     9
Delhi         8
Chennai       7
Mumbai        7
Kota          6
Hyderabad     6
Mysore        6
Kolkata       5
Pune          4
Bangalore     4
Gaya          2
Name: count, dtype: int64
--------------------------------------------------
occupation
retired           26
unemployed        15
government_job    14
student           12
freelancer        11
business_owner    11
private_job       11
Name: count, dtype: int64


In [72]:
df['bmi'] = df['weight'] / numpy.sqrt(df['height'])

In [73]:
df.head()

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category,bmi
0,67,119.8,1.56,2.92,False,Jaipur,retired,High,95.916764
1,36,101.1,1.83,34.28,False,Chennai,freelancer,Low,74.735270
2,39,56.8,1.64,36.64,False,Indore,freelancer,Low,44.353348
3,22,109.4,1.55,3.34,True,Mumbai,student,Medium,87.872195
4,69,62.2,1.60,3.94,True,Indore,retired,High,49.173418


In [74]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

In [92]:
preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', StandardScaler(), [0, 1, 2, 3, 7]),
        ('ordinal encoder', OrdinalEncoder(
            categories=[[False, True]]
        ), [4]),
        ("OHE", OneHotEncoder(drop='first', handle_unknown='ignore'), [5, 6])
    ],
    remainder='passthrough'
)

In [93]:
pipeline = Pipeline(
    steps=[
        ('processor', preprocessor)
    ]
)

In [94]:
from sklearn.model_selection import train_test_split

In [95]:
df.columns

Index(['age', 'weight', 'height', 'income_lpa', 'smoker', 'city', 'occupation',
       'insurance_premium_category', 'bmi'],
      dtype='str')

In [96]:
x = df.drop('insurance_premium_category', axis = 1)
y = df['insurance_premium_category']

In [97]:
x

,age,weight,height,income_lpa,smoker,city,occupation,bmi
0,67,119.8,1.56,2.92000,False,Jaipur,retired,95.916764
1,36,101.1,1.83,34.28000,False,Chennai,freelancer,74.735270
2,39,56.8,1.64,36.64000,False,Indore,freelancer,44.353348
3,22,109.4,1.55,3.34000,True,Mumbai,student,87.872195
4,69,62.2,1.60,3.94000,True,Indore,retired,49.173418
...,...,...,...,...,...,...,...,...
95,36,52.8,1.57,19.64000,False,Indore,business_owner,42.138987
96,26,113.8,1.54,34.01000,False,Delhi,private_job,91.702653
97,52,60.8,1.80,44.86000,False,Hyderabad,freelancer,45.317644
98,27,101.1,1.82,28.30000,False,Kolkata,business_owner,74.940306


In [98]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)

In [100]:
xTrain = pipeline.fit_transform(x_train)
xTest = pipeline.transform(x_test)

In [110]:
from sklearn.preprocessing import LabelEncoder

In [111]:
encoder = LabelEncoder()

yTrain = encoder.fit_transform(y_train)
yTest = encoder.transform(y_test)

In [102]:
import mlflow

In [103]:
mlflow.create_experiment("Run ALL ML Experiment")

2026/10/30 20:24:31 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/30 20:24:31 INFO mlflow.store.db.utils: Updating database tables


'1'

In [104]:
mlflow.set_experiment("Run ALL ML Experiment")

<Experiment: artifact_location='/Users/himmat/all-concept/notebook/mlruns/1', creation_time=1793371171536, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1793371171536, lifecycle_stage='active', name='Run ALL ML Experiment', tags={}, trace_location=None, workspace='default'>

In [117]:
models = {
    "Decision Tree": DecisionTreeClassifier(),
    "SVM": SVC(),
    "KNN": KNeighborsClassifier(),
    "Linear Regression": LogisticRegression(),
    "Naive Bayes": GaussianNB()
}

In [118]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

In [119]:
for name, model in models.items():
    with mlflow.start_run(run_name=name):
        model.fit(xTrain, yTrain)
        y_pred = model.predict(xTest)

        accuracy = accuracy_score(yTest, y_pred)
        precision = precision_score(yTest, y_pred, average='weighted', zero_division=0)
        recall = recall_score(yTest, y_pred, average='weighted', zero_division=0)
        f1 = f1_score(yTest, y_pred, average='weighted', zero_division=0)

        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        mlflow.sklearn.log_model(
            model,
            artifact_path="model",
            serialization_format="cloudpickle"
        )

2026/10/30 20:34:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/10/30 20:34:49 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/10/30 20:34:50 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/10/30 20:34:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/30 20:34:50 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' form